# Bronze ingestion — Chart of Accounts

`PoC_Charts_of_Accounts.xlsx` in GCS → `bronze_coa_raw.csv` in GCS → `bronze_coa_raw` in BigQuery.

**Expected result: 188 rows** (78 GROUP + 66 SABIC + 44 Petro Rabigh).

The parsing logic lives in the `bronze_ingest` package, not in these cells. That is
deliberate: Path 2 (a CFO uploading a file mid-chat) needs to call the same code from a
Cloud Run job, and it can only do that if the notebook stays a thin caller. If you find
yourself pasting parsing logic into a cell, stop.

Run order: 1 → 6. Cell 7 verifies the load and should be run every time.

## 1. Dependencies

Colab Enterprise (what BigQuery Studio runs) ships the GCP client libraries. `openpyxl` is not included.

In [ ]:
!pip install -q openpyxl

## 2. Get the package onto the runtime

Notebook runtimes are **ephemeral** — this runs every session, not once. Pick whichever
matches how the code is shared; the git option is better because it records which commit
produced a given load.

In [ ]:
# Option A — from a bucket
!gsutil -q cp -r gs://YOUR_BUCKET/code/bronze_ingest .

# Option B — from git (preferred: the commit is your provenance)
# !git clone -q https://YOUR_REPO/bronze_ingestion.git
# !pip install -q -e bronze_ingestion

import bronze_ingest
print("bronze_ingest", bronze_ingest.__version__)

## 3. Configuration

`LOCATION` is the one value you cannot change later. A BigQuery dataset's location is
**immutable**, and BigQuery **refuses to join across locations** — so bronze, silver and
anything the DS team creates must all share it. For an Aramco engagement this is a data
residency question, not a performance one: `me-central2` is Dammam. Confirm with Hussein.

The GCS bucket must be in the **same location as the dataset**. A cross-location load is a
hard error, not a slow path. (Coming from Azure this is the sharpest difference — there you
would cross regions freely and just pay egress.)

In [ ]:
PROJECT  = "your-project-id"          # TODO
LOCATION = "me-central2"              # TODO - must match the bucket
BUCKET   = "your-bucket"              # TODO - must be in LOCATION

DATASET  = "fcrd_bronze"
SRC_XLSX = f"gs://{BUCKET}/raw/PoC_Charts_of_Accounts.xlsx"
DST_CSV  = f"gs://{BUCKET}/bronze/bronze_coa_raw.csv"
TABLE_ID = f"{PROJECT}.{DATASET}.bronze_coa_raw"

# Parsing knobs. Note what is NOT here: no row numbers, no tab names, no column
# count. Those are discovered from the workbook, because those are what change.
CFG = {
    "header_sentinel": "account",
    "affiliate_pattern": r"\((\d{4})\)",
    "group_marker": "group",
    "drop_section_rows": True,
}

print(SRC_XLSX, "->", DST_CSV, "->", TABLE_ID)

## 4. Create the dataset and table

Run **once**. `CREATE TABLE IF NOT EXISTS` separates schema management from data
management, so re-running this cell can never drop data — only the load in cell 6
replaces rows.

The full annotated DDL, including column descriptions, is in `bronze_ddl.sql`. Column
descriptions are not decoration here: Agent 6 is text-to-SQL over this warehouse, and the
description is what tells it which column to pick.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT, location=LOCATION)

client.query(f'''
CREATE SCHEMA IF NOT EXISTS `{PROJECT}.{DATASET}`
OPTIONS (location = '{LOCATION}',
         description = 'FC&RD PoC - bronze layer. Raw landing, all STRING.')
''').result()

client.query(f'''
CREATE TABLE IF NOT EXISTS `{TABLE_ID}` (
  chart_scope      STRING OPTIONS (description = 'GROUP, 2010 or 2380. Added at ingest from the tab name. The two affiliate charts share 42 codes, 15 with different meanings - never join on account alone.'),
  account          STRING OPTIONS (description = 'Account code as text. Affiliates use 4-digit codes; the Group chart uses G-prefixed 5-digit nodes. Never empty.'),
  account_name     STRING OPTIONS (description = 'Account description, leading spaces preserved - indentation marks level-2 sub-accounts on the Group chart.'),
  statement        STRING OPTIONS (description = 'Balance sheet | Income statement. The trial balances use BS/PL; bronze does not harmonise them.'),
  category         STRING OPTIONS (description = 'FS caption group. 13 distinct values across the affiliate charts.'),
  normal_balance   STRING OPTIONS (description = 'Dr or Cr - the side this account normally sits on.'),
  level            STRING OPTIONS (description = 'Group chart only; empty for affiliates. 1 = face caption (52 nodes), 2 = sub-account (26).'),
  source_reference STRING OPTIONS (description = 'Group chart only; empty for affiliates. Traces to Aramco Q1-2026 interim report.')
)
OPTIONS (description = 'Charts of accounts for both affiliates and the Group, stacked. 188 rows = 78 + 66 + 44. Section-divider rows dropped at ingest. No relationships enforced. SYNTHETIC data.')
''').result()

print("dataset and table ready")

## 5. Extract and check

`extract()` reads the workbook straight from GCS bytes — `excel.py` never learns it is in
the cloud. It **raises** if any tab's `Total accounts: N` footer disagrees with the rows
parsed, and nothing is written when it does. A pipeline that lands unverified data is worse
than one that fails, because the failure is then found downstream by someone who trusts the
number.

In [ ]:
from bronze_ingest.coa import extract, to_csv_text
from bronze_ingest.cloud import read_workbook

report = []
rows, meta = extract(read_workbook(SRC_XLSX), CFG, report)
print("\n".join(report))

assert len(rows) == 188, f"expected 188 rows, got {len(rows)}"
csv_text = to_csv_text(rows)
print(f"\n{len(rows)} rows, {len(csv_text):,} bytes of CSV")

## 6. Land the CSV, then load it

The CSV in GCS is the **lineage artifact**, not a temp file — it is what proves, byte for
byte, what was loaded.

Two settings on the load carry all the risk:

- **`autodetect=False`** — autodetection would type `account` as `INT64`, making `G11000`
  and `1100` incompatible, and would silently break the all-STRING contract.
- **`WRITE_TRUNCATE`** — reloading the same pack with append gives 376 rows and *every
  check still passes proportionally*. Wrong but internally consistent is the failure mode
  to fear.

In [ ]:
from bronze_ingest.cloud import upload_csv, load_csv_to_bq

upload_csv(csv_text, DST_CSV)
job = load_csv_to_bq(DST_CSV, TABLE_ID, LOCATION)
print(f"loaded {job.output_rows} rows into {TABLE_ID}")

assert job.output_rows == len(rows), "load dropped or added rows"

## 7. Verify in BigQuery

"The CSV was right" and "the table is right" are different claims — the load is a step that
can lose data on its own. Run this every time.

**V2 is the one to watch.** BigQuery's CSV loader turns an empty unquoted field into `NULL`
by default; `cloud.py` sets `null_marker` so empties stay empty strings. That behaviour is
worth confirming rather than assuming. If V2 returns `nulls=110`, the marker did not apply
and every downstream `= ''` predicate needs `COALESCE`.

In [ ]:
q = lambda sql: client.query(sql).to_dataframe()

print("V1 row count - expect 188")
display(q(f"SELECT COUNT(*) AS rows FROM `{TABLE_ID}`"))

print("V2 empty vs NULL - expect nulls=0, empties=110")
display(q(f'''
SELECT COUNTIF(level IS NULL) AS nulls, COUNTIF(level = '') AS empties
FROM `{TABLE_ID}`
'''))

print("V3 scope breakdown - expect GROUP 78, 2010 66, 2380 44")
display(q(f"SELECT chart_scope, COUNT(*) AS rows FROM `{TABLE_ID}` GROUP BY 1 ORDER BY 1"))

print("V4 no empty account or name - expect 0, 0")
display(q(f'''
SELECT COUNTIF(COALESCE(account, '') = '') AS empty_account,
       COUNTIF(COALESCE(account_name, '') = '') AS empty_name
FROM `{TABLE_ID}`
'''))

## 8. The collision, made visible

Expect **15 rows**. Worth keeping loaded: it is the one-screen proof for why `dim_account`
must be keyed `(entity_code, account_code)`, and a good thing to have ready if the client
asks why the model looks the way it does.

`4010` is the nastiest — both sides are plausible revenue lines, so a join on code alone
returns a number that still foots.

In [ ]:
display(q(f'''
WITH a AS (
  SELECT chart_scope, account, TRIM(account_name) AS nm
  FROM `{TABLE_ID}`
  WHERE chart_scope IN ('2010', '2380')
)
SELECT s.account, s.nm AS sabic, p.nm AS petro_rabigh
FROM a s JOIN a p USING (account)
WHERE s.chart_scope = '2010' AND p.chart_scope = '2380' AND s.nm <> p.nm
ORDER BY s.account
'''))